# 04 - Benchmark + first baselines
Task: given a user's EARLIER messages, pick their real FUTURE reply out of 10 candidates
(1 real + 9 written by other people). Chance = 10%.
Needs scikit-learn:  pip install scikit-learn

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from src.benchmark.build import build_benchmark
from src.benchmark.evaluate import (StyleSpace, evaluate, summarize,
                                    learning_curve, DEFAULT_FEATURES)

df = pd.read_csv(ROOT / "data" / "processed" / "enron_messages.csv")
df["message_id"] = df["message_id"].astype(str)
FEATURES = [f for f in DEFAULT_FEATURES if (df[f] != 0).mean() >= 0.01]
print("features used:", FEATURES)
print("dropped (almost always zero):", [f for f in DEFAULT_FEATURES if f not in FEATURES])
print(df["user_split"].value_counts().to_dict(), df["time_split"].value_counts().to_dict())

features used: ['log_words', 'avg_sent_len', 'punct_density', 'n_questions', 'n_exclam', 'caps_ratio', 'starts_lower', 'has_newline']
dropped (almost always zero): []
{'train': 43158, 'val': 12629, 'test': 11918} {'history': 54201, 'eval': 13504}


## A. Build the benchmark (four difficulty tiers, same items in every tier)

In [2]:
bench = build_benchmark(df, n_neg=9, eval_splits=("val", "test"), max_items_per_user=50, seed=0)
bench.to_csv(ROOT / "data" / "processed" / "benchmark_v0.csv", index=False)
n_items = bench["item_id"].nunique()
print(f"{n_items} items from {bench['user_id'].nunique()} users, {len(bench)} lineups")
if n_items < 200:
    print("WARNING: few items -> noisy results. Consider eval_splits=('train','val','test') "
          "for the untrained baselines, or a lower MIN_MSGS in notebook 03.")

1179 items from 29 users, 4716 lineups


## B. Read some lineups BY EYE (is the benchmark sensible?)
The REAL reply and its negatives for the hardest tier.

In [3]:
by_id = df.set_index("message_id")
ex = bench[bench.tier == "content_and_length"].sample(3, random_state=1)
for item, neg in zip(ex["item_id"], ex["neg_ids"]):
    print("REAL    :", by_id.loc[item, "message"][:160].replace("\n", " "))
    for n in neg.split(";")[:3]:
        print("  other :", by_id.loc[n, "message"][:160].replace("\n", " "))
    print()

REAL    : Oh, Terrie, Ken won't be in town.  Otherwise, I'm sure he'd love to.  TERRIE JAMES@ENRON COMMUNICATIONS 04/16/2001 10:26 AM To: Kenneth Lay/Corp/Enron@ENRON, Je
  other : I think we should do this but should include some comments praising FERC for taking some positive actions and really looking for solutions instead of simply way
  other : No. We can wait until he gets back on the 1st.  Thanks.  Carol St. Clair EB 3889 713-853-3989 (Phone) 713-646-3393 (Fax) carol.st.clair@enron.com  Suzanne Adams
  other : I'm remembering now.  It was a term sheet which was never executed, as far as I know.  I believe FPL prepared it, and we may have commented on it. Let me see wh

REAL    : Audrey,  Please send about 6 maps the contact below.  Thanks, Kim.
  other : Don't forget.  Scott is still wanting a Salt Lake City contact.
  other : Allegheny's SC is now APX 5. Contact numbers are below.  Thanks Bill
  other : Please send me the Transaction referenced in the Termination.  Thanks.  Sar

## C. Baselines: recall@1 per tier (mean over users, 95% CI bootstraps USERS)
Read it like this:
- random should be ~0.10 everywhere (sanity check).
- user_length_only should DROP toward chance in the length-controlled tiers.
- user_style still above chance in 'content_and_length' = real style signal beyond length+topic.

In [4]:
space = StyleSpace(df, FEATURES)
res = evaluate(space, bench, seed=0)
summ = summarize(res)
summ.round(3)

,scorer,tier,users,recall@1,ci_low,ci_high,MRR
0,user_style,random,29,0.313,0.267,0.373,0.503
1,user_length_only,random,29,0.134,0.101,0.170,0.350
2,population,random,29,0.127,0.090,0.167,0.312
3,random,random,29,0.111,0.092,0.130,0.302
4,user_style,length_matched,29,0.257,0.203,0.316,0.452
5,random,length_matched,29,0.103,0.084,0.118,0.287
6,population,length_matched,29,0.097,0.072,0.124,0.290
7,user_length_only,length_matched,29,0.013,0.008,0.018,0.207
8,user_style,content_similar,29,0.239,0.196,0.289,0.439
9,user_length_only,content_similar,29,0.115,0.089,0.143,0.324


## D. Learning curve: how much history does personalization need?
0 history = chance by construction (nothing personal to go on).

In [5]:
lc = learning_curve(space, bench)
lc.round(3)

tier,random,length_matched,content_similar,content_and_length
history_msgs,,,,
0,0.100,0.100,0.100,0.100
1,0.184,0.154,0.157,0.145
3,0.214,0.193,0.177,0.158
10,0.254,0.221,0.201,0.181
30,0.291,0.248,0.225,0.204
100,0.293,0.257,0.218,0.205
all,0.306,0.247,0.227,0.216


In [6]:
from src.benchmark.build_v1 import build_benchmark_v1
from src.benchmark.audit import lineup_attacks, paired_gain
findings_dir = ROOT / "docs" / "findings"
findings_dir.mkdir(parents=True, exist_ok=True)

bench_v1 = build_benchmark_v1(
    df, n_neg=9, eval_splits=("val", "test"), max_items_per_user=50, seed=0
)
bench_v1.to_csv(ROOT / "data" / "processed" / "benchmark_v1.csv", index=False)
print(f"v1: {bench_v1['item_id'].nunique()} items, {bench_v1['user_id'].nunique()} users, {len(bench_v1)} lineups")

audit_v0_raw = lineup_attacks(df, bench, space)
audit_v1_raw = lineup_attacks(df, bench_v1, space)
audit_v0_by_user = audit_v0_raw.groupby(["scorer", "tier", "user_id"], as_index=False)["hit"].mean()
audit_v1_by_user = audit_v1_raw.groupby(["scorer", "tier", "user_id"], as_index=False)["hit"].mean()
audit_v0 = audit_v0_by_user.groupby(["scorer", "tier"], as_index=False)["hit"].mean().rename(columns={"hit": "v0_recall@1"})
audit_v1 = audit_v1_by_user.groupby(["scorer", "tier"], as_index=False)["hit"].mean().rename(columns={"hit": "v1_recall@1"})
audit_comparison = audit_v0.merge(audit_v1, on=["scorer", "tier"], validate="one_to_one")
audit_comparison.to_csv(findings_dir / "04_v1_lineup_audit.csv", index=False)
print("Lineup attacks, recall@1 (mean over users):")
print(audit_comparison.round(3).to_string(index=False))

v1: 589 items, 28 users, 2356 lineups


Lineup attacks, recall@1 (mean over users):
               scorer               tier  v0_recall@1  v1_recall@1
lineup_content_center content_and_length        0.555        0.061
lineup_content_center    content_similar        0.595        0.053
lineup_content_center     length_matched        0.088        0.080
lineup_content_center             random        0.087        0.064
 lineup_length_center content_and_length        0.220        0.107
 lineup_length_center    content_similar        0.088        0.141
 lineup_length_center     length_matched        0.220        0.099
 lineup_length_center             random        0.127        0.115
  lineup_style_center content_and_length        0.106        0.071
  lineup_style_center    content_similar        0.090        0.096
  lineup_style_center     length_matched        0.088        0.085
  lineup_style_center             random        0.117        0.084


In [7]:
res_v1 = evaluate(space, bench_v1, seed=0)
summary_v1 = summarize(res_v1)
summary_v1.to_csv(ROOT / "docs" / "findings" / "04_v1_baselines.csv", index=False)
print("v1 baselines:")
print(summary_v1.round(3).to_string(index=False))

v1 baselines:
          scorer               tier  users  recall@1  ci_low  ci_high   MRR
      user_style             random     28     0.291   0.225    0.364 0.476
user_length_only             random     28     0.130   0.086    0.187 0.328
          random             random     28     0.120   0.090    0.151 0.306
      population             random     28     0.090   0.055    0.134 0.279
      user_style     length_matched     28     0.245   0.180    0.316 0.446
      population     length_matched     28     0.121   0.078    0.167 0.303
user_length_only     length_matched     28     0.104   0.087    0.121 0.276
          random     length_matched     28     0.088   0.064    0.115 0.275
      user_style    content_similar     28     0.221   0.168    0.276 0.421
      population    content_similar     28     0.116   0.080    0.154 0.319
          random    content_similar     28     0.094   0.070    0.117 0.290
user_length_only    content_similar     28     0.091   0.063    0.118 0.29

In [8]:
paired_gain_v1 = paired_gain(res_v1)
learning_curve_v1 = learning_curve(space, bench_v1)
paired_gain_v1.to_csv(ROOT / "docs" / "findings" / "04_v1_paired_gain.csv", index=False)
learning_curve_v1.to_csv(ROOT / "docs" / "findings" / "04_v1_learning_curve.csv")
print("Paired gain (user_style minus population):")
print(paired_gain_v1.round(3).to_string(index=False))
print("Learning curve:")
print(learning_curve_v1.round(3).to_string())

Paired gain (user_style minus population):
              tier  users  mean_gain  ci_low  ci_high  share_users_improved
            random     28      0.201   0.117    0.289                 0.786
    length_matched     28      0.124   0.044    0.216                 0.536
   content_similar     28      0.106   0.039    0.174                 0.643
content_and_length     28      0.095   0.034    0.156                 0.571
Learning curve:
tier          random  length_matched  content_similar  content_and_length
history_msgs                                                             
0              0.100           0.100            0.100               0.100
1              0.178           0.160            0.165               0.165
3              0.241           0.206            0.186               0.177
10             0.294           0.249            0.198               0.180
30             0.324           0.277            0.225               0.201
100            0.320           0.273       